# 02a — Cache activations from trained transformer

Owner: Cole S.

In [1]:
import sys
import torch
sys.path.insert(0, "../src")

from mid.config import load_configs
from mid.model.hooked_model import load_checkpoint
from mid.sae.activations import cache_activations, read_activations

Set Parameters in the cell below to change which model is loaded and/or what information is grabbed from the hooked transformers

In [2]:
#Model type: String: [decoder, encoder, or encoderdecoder]  Default: decoder
model_type = "decoder"
#Model size: String: [small, medium, large]  Default: small 
model_size = "small"

#Hook type: String: [mlp, attention, stream]  Default: stream
#Specifies what information to grad from the hooked transformers to be processed for input into the SAE.
# stream : Grabs the residual stream 
# attention : Grabs the attention heads
# mlp : Grabs the multilayer perceptron activations
hook_type = "stream"

#Use split: Bool: [True, False]  Default: True 
#Whether to split data into train and validation for the SAE or to use all data. 
use_split = True

#Print the paths
config_path = f"../configs/model_{model_type}_{model_size}.yaml"
checkpoint_path = f"../checkpoints/{model_type}_{model_size}_best_model.pt"
out_path = f"../data/sae_data/{model_type}_{model_size}"
get_path = f"../data/sae_data/{model_type}_{model_size}/train.pt"

print(config_path)
print(checkpoint_path)
print(out_path)

../configs/model_decoder_small.yaml
../checkpoints/decoder_small_best_model.pt
../data/sae_data/decoder_small


In [3]:
model_cfg, train_cfg = load_configs(config_path)
batch_size = train_cfg.batch_size
print(model_cfg)

ModelConfig(d_model=128, d_head=32, n_heads=4, n_layers=4, d_mlp=512, d_vocab=3000, n_ctx=256, act_fn='gelu', normalization_type='LN', tokenizer_name=None, seed=16)


In [4]:
#DEBUG: Load model
model = load_checkpoint(checkpoint_path, model_cfg)

Moving model to device:  cuda
Model parameters: 1,597,112
Moving model to device:  cuda


In [5]:
#Cache activations for SAE input
cache_activations(checkpoint_path, model_cfg, hook_type, out_path, batch_size, use_split)

Moving model to device:  cuda
Model parameters: 1,597,112
Moving model to device:  cuda
Processed batch 1 out of 92 for train
Processed batch 51 out of 92 for train
Saved activations by layer to ..\data\sae_data\decoder_small\train.pt
Processed batch 1 out of 11 for val
Saved activations by layer to ..\data\sae_data\decoder_small\val.pt


In [6]:
#DEBUG: Read a train.pt file's first layer.
activations, metadata, num_tokens, num_activations = read_activations(get_path, 0)
print("Activations: ", activations)
print("Metadata: ", metadata)
print("Num Tokens: ", num_tokens)
print("Num Activation: ", num_activations)

Activations:  tensor([[ 1.1435, -0.0707, -0.8078,  ...,  0.4470,  1.2245, -1.0420],
        [-0.2160,  1.8079, -0.8403,  ..., -0.0577,  1.9813, -2.3543],
        [ 0.3348,  0.2121, -1.8232,  ..., -0.1346,  0.7225, -1.0614],
        ...,
        [ 1.1149,  0.2480,  0.9998,  ...,  0.6680, -0.8065,  0.6201],
        [ 0.3798,  0.2976,  1.2421,  ...,  0.4944,  0.3431,  0.8312],
        [ 0.7420, -0.0245,  0.8454,  ...,  1.0054, -0.3514,  0.2785]])
Metadata:  {'token_ids': tensor([2410,   79,  825,  ...,  120,  146,   12]), 'seq_idx': tensor([   0,    0,    0,  ..., 5832, 5832, 5832]), 'pos_idx': tensor([  0,   1,   2,  ..., 253, 254, 255]), 'split': 'train', 'seq_len': 256, 'hook_type': 'stream'}
Num Tokens:  1493248
Num Activation:  128
